In [1]:
import sys
sys.path.append("../src")
from storage import load_database, DATA_DIR
from queries import count_by_type , history_for_procedure, linked_partner, linked_pairs, last_completed_all, procedure_due_dates, overdue_procedures, due_next, due_within, overdue_by_location

In [2]:
conn = load_database(data_dir=DATA_DIR / 'generated')
for t in ['Frequencies', 'Assets', 'Procedures', 'Service_History']:
    print(t, conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0])

Frequencies 4
Assets 40
Procedures 160
Service_History 543


In [3]:
cur = conn.cursor()

count_by_type prototype:

In [4]:
proto_rows = cur.execute("SELECT type, COUNT(*) FROM Assets GROUP BY type ORDER BY COUNT(*) DESC, type").fetchall()
for asset_type, count in proto_rows:
    print(f'{asset_type}: {count}')

generator: 9
motor: 8
pump: 8
transformer: 8
boiler: 4
hvac: 3


In [5]:
rows = count_by_type(conn)
for asset_type, count in rows:
    print(f'{asset_type}: {count}')

generator: 9
motor: 8
pump: 8
transformer: 8
boiler: 4
hvac: 3


In [6]:
assert(proto_rows == count_by_type(conn))

history_for_procedure prototype:

In [7]:
proc_id = 'PMP-002-P1'

sql = '''
    SELECT record_id, date_completed, notes FROM Service_History
    WHERE procedure_id = ?
    ORDER BY date_completed
    '''
proto_rows = cur.execute(sql, (proc_id,)).fetchall()
for record_id, date_completed, notes in proto_rows:
    print(f'{record_id}, {date_completed}, {notes}')

SH-0016, 2026-08-01, None
SH-0017, 2026-08-08, None
SH-0018, 2026-08-17, None
SH-0019, 2026-08-24, None
SH-0020, 2026-08-30, None
SH-0021, 2026-09-05, None
SH-0022, 2026-09-13, None
SH-0023, 2026-09-23, None
SH-0024, 2026-09-26, None
SH-0025, 2026-10-03, None
SH-0026, 2026-10-13, None
SH-0027, 2026-10-17, None
SH-0028, 2026-10-28, None


In [8]:
rows = history_for_procedure(conn ,'PMP-002-P1')
for record_id, date_completed, notes in rows:
    print(f'{record_id}, {date_completed}, {notes}')

SH-0016, 2026-08-01, None
SH-0017, 2026-08-08, None
SH-0018, 2026-08-17, None
SH-0019, 2026-08-24, None
SH-0020, 2026-08-30, None
SH-0021, 2026-09-05, None
SH-0022, 2026-09-13, None
SH-0023, 2026-09-23, None
SH-0024, 2026-09-26, None
SH-0025, 2026-10-03, None
SH-0026, 2026-10-13, None
SH-0027, 2026-10-17, None
SH-0028, 2026-10-28, None


In [9]:
assert(proto_rows == history_for_procedure(conn ,'PMP-002-P1'))

linked_partner prototype:

In [10]:
asset_id = 'PMP-001'

sql = '''
    SELECT a.asset_id, a.type, b.asset_id, b.type 
    FROM Assets as a
    JOIN Assets as b
    ON a.linked_asset_id = b.asset_id
    WHERE a.asset_id = ?
    ORDER BY a.asset_id
    '''
proto_rows = cur.execute(sql, (asset_id,)).fetchall()
for a_id, a_type, b_id, b_type in proto_rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

PMP-001, pump, MTR-001, motor


In [11]:
rows = linked_partner(conn, 'PMP-001')
for a_id, a_type, b_id, b_type in rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

PMP-001, pump, MTR-001, motor


In [12]:
assert(proto_rows == linked_partner(conn ,'PMP-001'))

linked_pairs prototype:

In [13]:
sql = '''
    SELECT a.asset_id, a.type, b.asset_id, b.type 
    FROM Assets as a
    JOIN Assets as b
    ON a.linked_asset_id = b.asset_id
    WHERE a.asset_id < b.asset_id
    ORDER BY a.asset_id
    '''
proto_rows = cur.execute(sql).fetchall()
for a_id, a_type, b_id, b_type in proto_rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

MTR-001, motor, PMP-001, pump
MTR-002, motor, PMP-002, pump
MTR-003, motor, PMP-003, pump
MTR-004, motor, PMP-004, pump
MTR-005, motor, PMP-005, pump
MTR-006, motor, PMP-006, pump
MTR-007, motor, PMP-007, pump
MTR-008, motor, PMP-008, pump


In [14]:
rows = linked_pairs(conn)
for a_id, a_type, b_id, b_type in rows:
    print(f'{a_id}, {a_type}, {b_id}, {b_type}')

MTR-001, motor, PMP-001, pump
MTR-002, motor, PMP-002, pump
MTR-003, motor, PMP-003, pump
MTR-004, motor, PMP-004, pump
MTR-005, motor, PMP-005, pump
MTR-006, motor, PMP-006, pump
MTR-007, motor, PMP-007, pump
MTR-008, motor, PMP-008, pump


In [15]:
assert(proto_rows == linked_pairs(conn))

last_completed_all prototype:

In [16]:
sql = '''
    SELECT p.procedure_id, h.last_done
    FROM Procedures as p
    LEFT JOIN  ( 
        SELECT procedure_id, MAX(date_completed)
    AS last_done
        FROM Service_History
        GROUP BY procedure_id
    ) as h
    ON p.procedure_id = h.procedure_id
    ORDER BY p.procedure_id
    '''
proto_rows = cur.execute(sql).fetchall()
print(len(proto_rows))
for last_proc, last_done in proto_rows:
    print(f'{last_proc}, {last_done}')

160
BLR-001-P1, 2026-10-28
BLR-001-P2, 2026-10-21
BLR-001-P3, None
BLR-001-P4, None
BLR-002-P1, 2026-10-29
BLR-002-P2, 2026-09-06
BLR-002-P3, 2026-08-10
BLR-002-P4, None
BLR-003-P1, None
BLR-003-P2, 2026-10-02
BLR-003-P3, 2026-09-04
BLR-003-P4, None
BLR-004-P1, 2026-10-24
BLR-004-P2, None
BLR-004-P3, 2026-10-08
BLR-004-P4, None
GEN-001-P1, None
GEN-001-P2, 2026-08-21
GEN-001-P3, 2026-10-07
GEN-001-P4, None
GEN-002-P1, 2026-10-29
GEN-002-P2, 2026-10-24
GEN-002-P3, None
GEN-002-P4, None
GEN-003-P1, 2026-09-30
GEN-003-P2, 2026-08-25
GEN-003-P3, 2026-10-29
GEN-003-P4, None
GEN-004-P1, 2026-10-29
GEN-004-P2, 2026-10-04
GEN-004-P3, None
GEN-004-P4, None
GEN-005-P1, 2026-10-11
GEN-005-P2, 2026-10-04
GEN-005-P3, 2026-10-09
GEN-005-P4, None
GEN-006-P1, 2026-10-09
GEN-006-P2, 2026-10-21
GEN-006-P3, 2026-09-04
GEN-006-P4, None
GEN-007-P1, 2026-10-29
GEN-007-P2, 2026-10-13
GEN-007-P3, 2026-08-28
GEN-007-P4, None
GEN-008-P1, 2026-10-28
GEN-008-P2, 2026-10-28
GEN-008-P3, 2026-08-16
GEN-008-P4, None


In [17]:
rows = last_completed_all(conn)
print(len(rows))
for last_proc, last_done in rows:
    print(f'{last_proc}, {last_done}')

160
BLR-001-P1, 2026-10-28
BLR-001-P2, 2026-10-21
BLR-001-P3, None
BLR-001-P4, None
BLR-002-P1, 2026-10-29
BLR-002-P2, 2026-09-06
BLR-002-P3, 2026-08-10
BLR-002-P4, None
BLR-003-P1, None
BLR-003-P2, 2026-10-02
BLR-003-P3, 2026-09-04
BLR-003-P4, None
BLR-004-P1, 2026-10-24
BLR-004-P2, None
BLR-004-P3, 2026-10-08
BLR-004-P4, None
GEN-001-P1, None
GEN-001-P2, 2026-08-21
GEN-001-P3, 2026-10-07
GEN-001-P4, None
GEN-002-P1, 2026-10-29
GEN-002-P2, 2026-10-24
GEN-002-P3, None
GEN-002-P4, None
GEN-003-P1, 2026-09-30
GEN-003-P2, 2026-08-25
GEN-003-P3, 2026-10-29
GEN-003-P4, None
GEN-004-P1, 2026-10-29
GEN-004-P2, 2026-10-04
GEN-004-P3, None
GEN-004-P4, None
GEN-005-P1, 2026-10-11
GEN-005-P2, 2026-10-04
GEN-005-P3, 2026-10-09
GEN-005-P4, None
GEN-006-P1, 2026-10-09
GEN-006-P2, 2026-10-21
GEN-006-P3, 2026-09-04
GEN-006-P4, None
GEN-007-P1, 2026-10-29
GEN-007-P2, 2026-10-13
GEN-007-P3, 2026-08-28
GEN-007-P4, None
GEN-008-P1, 2026-10-28
GEN-008-P2, 2026-10-28
GEN-008-P3, 2026-08-16
GEN-008-P4, None


In [18]:
assert(proto_rows == last_completed_all(conn))

procedure_due_dates prototype:

In [19]:
sql = '''
    SELECT p.procedure_id, h.last_done, a.commissioning_date, f.service_interval_days, date(COALESCE(h.last_done, a.commissioning_date),'+' || f.service_interval_days || ' days') AS next_due
    FROM Procedures as p
    LEFT JOIN  ( 
        SELECT procedure_id, MAX(date_completed) AS last_done
        FROM Service_History
        GROUP BY procedure_id
    ) as h
    ON p.procedure_id = h.procedure_id
    JOIN Assets AS a ON p.asset_id = a.asset_id
    JOIN Frequencies AS f ON p.service_frequency = f.service_frequency
    ORDER BY p.procedure_id
    '''
proto_rows = cur.execute(sql).fetchall()
print(len(proto_rows))
for last_proc, last_done, commissioned, interval_days, next_due in proto_rows:
    print(f'{last_proc}, {last_done}, {commissioned}, {interval_days}, {next_due}')

160
BLR-001-P1, 2026-10-28, 2023-03-24, 7, 2026-11-04
BLR-001-P2, 2026-10-21, 2023-03-24, 30, 2026-11-20
BLR-001-P3, None, 2023-03-24, 90, 2023-06-22
BLR-001-P4, None, 2023-03-24, 365, 2024-03-23
BLR-002-P1, 2026-10-29, 2024-05-18, 7, 2026-11-05
BLR-002-P2, 2026-09-06, 2024-05-18, 30, 2026-10-06
BLR-002-P3, 2026-08-10, 2024-05-18, 90, 2026-11-08
BLR-002-P4, None, 2024-05-18, 365, 2025-05-18
BLR-003-P1, None, 2024-08-09, 7, 2024-08-16
BLR-003-P2, 2026-10-02, 2024-08-09, 30, 2026-11-01
BLR-003-P3, 2026-09-04, 2024-08-09, 90, 2026-12-03
BLR-003-P4, None, 2024-08-09, 365, 2025-08-09
BLR-004-P1, 2026-10-24, 2025-05-03, 7, 2026-10-31
BLR-004-P2, None, 2025-05-03, 30, 2025-06-02
BLR-004-P3, 2026-10-08, 2025-05-03, 90, 2027-01-06
BLR-004-P4, None, 2025-05-03, 365, 2026-05-03
GEN-001-P1, None, 2022-12-27, 7, 2023-01-03
GEN-001-P2, 2026-08-21, 2022-12-27, 30, 2026-09-20
GEN-001-P3, 2026-10-07, 2022-12-27, 90, 2027-01-05
GEN-001-P4, None, 2022-12-27, 365, 2023-12-27
GEN-002-P1, 2026-10-29, 2022-1

In [20]:
rows = procedure_due_dates(conn)
print(len(rows))
for last_proc, last_done, commissioned, interval_days, next_due in rows:
    print(f'{last_proc}, {last_done}, {commissioned}, {interval_days}, {next_due}')

160
BLR-001-P1, 2026-10-28, 2023-03-24, 7, 2026-11-04
BLR-001-P2, 2026-10-21, 2023-03-24, 30, 2026-11-20
BLR-001-P3, None, 2023-03-24, 90, 2023-06-22
BLR-001-P4, None, 2023-03-24, 365, 2024-03-23
BLR-002-P1, 2026-10-29, 2024-05-18, 7, 2026-11-05
BLR-002-P2, 2026-09-06, 2024-05-18, 30, 2026-10-06
BLR-002-P3, 2026-08-10, 2024-05-18, 90, 2026-11-08
BLR-002-P4, None, 2024-05-18, 365, 2025-05-18
BLR-003-P1, None, 2024-08-09, 7, 2024-08-16
BLR-003-P2, 2026-10-02, 2024-08-09, 30, 2026-11-01
BLR-003-P3, 2026-09-04, 2024-08-09, 90, 2026-12-03
BLR-003-P4, None, 2024-08-09, 365, 2025-08-09
BLR-004-P1, 2026-10-24, 2025-05-03, 7, 2026-10-31
BLR-004-P2, None, 2025-05-03, 30, 2025-06-02
BLR-004-P3, 2026-10-08, 2025-05-03, 90, 2027-01-06
BLR-004-P4, None, 2025-05-03, 365, 2026-05-03
GEN-001-P1, None, 2022-12-27, 7, 2023-01-03
GEN-001-P2, 2026-08-21, 2022-12-27, 30, 2026-09-20
GEN-001-P3, 2026-10-07, 2022-12-27, 90, 2027-01-05
GEN-001-P4, None, 2022-12-27, 365, 2023-12-27
GEN-002-P1, 2026-10-29, 2022-1

In [21]:
assert(proto_rows == procedure_due_dates(conn))

overdue_procedures prototype:

In [22]:
sql = '''
    SELECT d.procedure_id, d.last_done, d.next_due
    FROM (
        SELECT p.procedure_id, h.last_done, a.commissioning_date, f.service_interval_days, date(COALESCE(h.last_done, a.commissioning_date),'+' || f.service_interval_days || ' days') AS next_due
        FROM Procedures as p
        LEFT JOIN  ( 
            SELECT procedure_id, MAX(date_completed) AS last_done
            FROM Service_History
            GROUP BY procedure_id
        ) as h
        ON p.procedure_id = h.procedure_id
        JOIN Assets AS a ON p.asset_id = a.asset_id
        JOIN Frequencies AS f ON p.service_frequency = f.service_frequency
    ) AS d
    WHERE d.next_due < ?
    ORDER BY d.next_due, d.procedure_id
    '''
proto_rows = cur.execute(sql, ('2026-11-02',)).fetchall()
print(len(proto_rows))
for proc_id, last_done, next_due in proto_rows:
    print(f'{proc_id}, {last_done}, {next_due}')

89
GEN-001-P1, None, 2023-01-03
GEN-002-P3, None, 2023-02-28
PMP-008-P3, None, 2023-06-06
PMP-001-P1, None, 2023-06-18
BLR-001-P3, None, 2023-06-22
PMP-001-P3, None, 2023-09-09
PMP-005-P3, None, 2023-11-24
GEN-002-P4, None, 2023-11-30
GEN-001-P4, None, 2023-12-27
GEN-004-P3, None, 2023-12-31
MTR-006-P4, None, 2024-01-14
PMP-008-P4, None, 2024-03-07
BLR-001-P4, None, 2024-03-23
MTR-005-P4, None, 2024-04-11
GEN-003-P4, None, 2024-04-30
PMP-001-P4, None, 2024-06-10
TRN-008-P4, None, 2024-06-19
HVC-001-P1, None, 2024-07-11
TRN-002-P4, None, 2024-07-25
TRN-004-P2, None, 2024-08-10
GEN-007-P4, None, 2024-08-15
BLR-003-P1, None, 2024-08-16
PMP-005-P4, None, 2024-08-25
TRN-007-P4, None, 2024-09-21
GEN-004-P4, None, 2024-10-01
MTR-007-P3, None, 2024-11-10
MTR-008-P3, None, 2024-11-30
MTR-003-P3, None, 2025-01-23
BLR-002-P4, None, 2025-05-18
BLR-004-P2, None, 2025-06-02
PMP-002-P3, None, 2025-06-07
TRN-001-P3, None, 2025-06-28
HVC-001-P4, None, 2025-07-04
TRN-004-P4, None, 2025-07-11
BLR-003-P4,

In [23]:
rows = overdue_procedures(conn, '2026-11-02')
print(len(rows))
for proc_id, last_done, next_due in rows:
    print(f'{proc_id}, {last_done}, {next_due}')

89
GEN-001-P1, None, 2023-01-03
GEN-002-P3, None, 2023-02-28
PMP-008-P3, None, 2023-06-06
PMP-001-P1, None, 2023-06-18
BLR-001-P3, None, 2023-06-22
PMP-001-P3, None, 2023-09-09
PMP-005-P3, None, 2023-11-24
GEN-002-P4, None, 2023-11-30
GEN-001-P4, None, 2023-12-27
GEN-004-P3, None, 2023-12-31
MTR-006-P4, None, 2024-01-14
PMP-008-P4, None, 2024-03-07
BLR-001-P4, None, 2024-03-23
MTR-005-P4, None, 2024-04-11
GEN-003-P4, None, 2024-04-30
PMP-001-P4, None, 2024-06-10
TRN-008-P4, None, 2024-06-19
HVC-001-P1, None, 2024-07-11
TRN-002-P4, None, 2024-07-25
TRN-004-P2, None, 2024-08-10
GEN-007-P4, None, 2024-08-15
BLR-003-P1, None, 2024-08-16
PMP-005-P4, None, 2024-08-25
TRN-007-P4, None, 2024-09-21
GEN-004-P4, None, 2024-10-01
MTR-007-P3, None, 2024-11-10
MTR-008-P3, None, 2024-11-30
MTR-003-P3, None, 2025-01-23
BLR-002-P4, None, 2025-05-18
BLR-004-P2, None, 2025-06-02
PMP-002-P3, None, 2025-06-07
TRN-001-P3, None, 2025-06-28
HVC-001-P4, None, 2025-07-04
TRN-004-P4, None, 2025-07-11
BLR-003-P4,

In [24]:
assert(proto_rows == overdue_procedures(conn, '2026-11-02'))

due_next prototype:

In [25]:
sql = '''
    SELECT d.procedure_id, d.last_done, d.next_due
    FROM (
        SELECT p.procedure_id, h.last_done, a.commissioning_date, f.service_interval_days, date(COALESCE(h.last_done, a.commissioning_date),'+' || f.service_interval_days || ' days') AS next_due
        FROM Procedures as p
        LEFT JOIN  ( 
            SELECT procedure_id, MAX(date_completed) AS last_done
            FROM Service_History
            GROUP BY procedure_id
        ) as h
        ON p.procedure_id = h.procedure_id
        JOIN Assets AS a ON p.asset_id = a.asset_id
        JOIN Frequencies AS f ON p.service_frequency = f.service_frequency
    ) AS d
    ORDER BY d.next_due, d.procedure_id
    LIMIT ?
    '''
proto_rows = cur.execute(sql, (10,)).fetchall()
print(len(proto_rows))
for proc_id, last_done, next_due in proto_rows:
    print(f'{proc_id}, {last_done}, {next_due}')

10
GEN-001-P1, None, 2023-01-03
GEN-002-P3, None, 2023-02-28
PMP-008-P3, None, 2023-06-06
PMP-001-P1, None, 2023-06-18
BLR-001-P3, None, 2023-06-22
PMP-001-P3, None, 2023-09-09
PMP-005-P3, None, 2023-11-24
GEN-002-P4, None, 2023-11-30
GEN-001-P4, None, 2023-12-27
GEN-004-P3, None, 2023-12-31


In [26]:
rows = due_next(conn, 10)
print(len(rows))
for proc_id, last_done, next_due in rows:
    print(f'{proc_id}, {last_done}, {next_due}')

10
GEN-001-P1, None, 2023-01-03
GEN-002-P3, None, 2023-02-28
PMP-008-P3, None, 2023-06-06
PMP-001-P1, None, 2023-06-18
BLR-001-P3, None, 2023-06-22
PMP-001-P3, None, 2023-09-09
PMP-005-P3, None, 2023-11-24
GEN-002-P4, None, 2023-11-30
GEN-001-P4, None, 2023-12-27
GEN-004-P3, None, 2023-12-31


In [27]:
assert(proto_rows == due_next(conn, 10))

due_within prototype:

In [28]:
sql = '''
    SELECT d.procedure_id, d.last_done, d.next_due
    FROM (
        SELECT p.procedure_id, h.last_done, a.commissioning_date, f.service_interval_days, date(COALESCE(h.last_done, a.commissioning_date),'+' || f.service_interval_days || ' days') AS next_due
        FROM Procedures as p
        LEFT JOIN  ( 
            SELECT procedure_id, MAX(date_completed) AS last_done
            FROM Service_History
            GROUP BY procedure_id
        ) as h
        ON p.procedure_id = h.procedure_id
        JOIN Assets AS a ON p.asset_id = a.asset_id
        JOIN Frequencies AS f ON p.service_frequency = f.service_frequency
    ) AS d
    WHERE d.next_due >= ?
      AND d.next_due <= date(?, '+' || ? || ' days')
    ORDER BY d.next_due, d.procedure_id
    '''
proto_rows = cur.execute(sql, ('2026-11-02', '2026-11-02', 10)).fetchall()
print(len(proto_rows))
for proc_id, last_done, next_due in proto_rows:
    print(f'{proc_id}, {last_done}, {next_due}')

28
GEN-009-P1, 2026-10-26, 2026-11-02
HVC-002-P1, 2026-10-26, 2026-11-02
PMP-003-P3, 2026-08-04, 2026-11-02
GEN-004-P2, 2026-10-04, 2026-11-03
GEN-005-P2, 2026-10-04, 2026-11-03
MTR-004-P1, 2026-10-27, 2026-11-03
MTR-006-P2, 2026-10-04, 2026-11-03
BLR-001-P1, 2026-10-28, 2026-11-04
GEN-008-P1, 2026-10-28, 2026-11-04
PMP-002-P1, 2026-10-28, 2026-11-04
PMP-008-P1, 2026-10-28, 2026-11-04
TRN-005-P1, 2026-10-28, 2026-11-04
TRN-006-P1, 2026-10-28, 2026-11-04
BLR-002-P1, 2026-10-29, 2026-11-05
GEN-002-P1, 2026-10-29, 2026-11-05
GEN-004-P1, 2026-10-29, 2026-11-05
GEN-007-P1, 2026-10-29, 2026-11-05
MTR-002-P3, 2026-08-07, 2026-11-05
MTR-006-P1, 2026-10-29, 2026-11-05
TRN-007-P1, 2026-10-29, 2026-11-05
TRN-008-P1, 2026-10-29, 2026-11-05
MTR-001-P4, None, 2026-11-07
BLR-002-P3, 2026-08-10, 2026-11-08
HVC-002-P3, 2026-08-10, 2026-11-08
TRN-006-P2, 2026-10-09, 2026-11-08
GEN-007-P2, 2026-10-13, 2026-11-12
PMP-003-P2, 2026-10-13, 2026-11-12
TRN-001-P2, 2026-10-13, 2026-11-12


In [29]:
rows = due_within(conn, '2026-11-02', 10)
print(len(rows))
for proc_id, last_done, next_due in rows:
    print(f'{proc_id}, {last_done}, {next_due}')

28
GEN-009-P1, 2026-10-26, 2026-11-02
HVC-002-P1, 2026-10-26, 2026-11-02
PMP-003-P3, 2026-08-04, 2026-11-02
GEN-004-P2, 2026-10-04, 2026-11-03
GEN-005-P2, 2026-10-04, 2026-11-03
MTR-004-P1, 2026-10-27, 2026-11-03
MTR-006-P2, 2026-10-04, 2026-11-03
BLR-001-P1, 2026-10-28, 2026-11-04
GEN-008-P1, 2026-10-28, 2026-11-04
PMP-002-P1, 2026-10-28, 2026-11-04
PMP-008-P1, 2026-10-28, 2026-11-04
TRN-005-P1, 2026-10-28, 2026-11-04
TRN-006-P1, 2026-10-28, 2026-11-04
BLR-002-P1, 2026-10-29, 2026-11-05
GEN-002-P1, 2026-10-29, 2026-11-05
GEN-004-P1, 2026-10-29, 2026-11-05
GEN-007-P1, 2026-10-29, 2026-11-05
MTR-002-P3, 2026-08-07, 2026-11-05
MTR-006-P1, 2026-10-29, 2026-11-05
TRN-007-P1, 2026-10-29, 2026-11-05
TRN-008-P1, 2026-10-29, 2026-11-05
MTR-001-P4, None, 2026-11-07
BLR-002-P3, 2026-08-10, 2026-11-08
HVC-002-P3, 2026-08-10, 2026-11-08
TRN-006-P2, 2026-10-09, 2026-11-08
GEN-007-P2, 2026-10-13, 2026-11-12
PMP-003-P2, 2026-10-13, 2026-11-12
TRN-001-P2, 2026-10-13, 2026-11-12


In [30]:
assert(proto_rows == due_within(conn, '2026-11-02', 10))

overdue_by_location prototype:

In [31]:
sql = '''
    SELECT d.location, SUM(CASE WHEN d.last_done IS NOT NULL THEN 1 ELSE 0 END) AS overdue_with_history, SUM(CASE WHEN d.last_done IS NULL THEN 1 ELSE 0 END) AS overdue_never_serviced
    FROM (
        SELECT p.procedure_id, h.last_done, a.commissioning_date, a.location, f.service_interval_days, date(COALESCE(h.last_done, a.commissioning_date),'+' || f.service_interval_days || ' days') AS next_due
        FROM Procedures as p
        LEFT JOIN  ( 
            SELECT procedure_id, MAX(date_completed) AS last_done
            FROM Service_History
            GROUP BY procedure_id
        ) as h
        ON p.procedure_id = h.procedure_id
        JOIN Assets AS a ON p.asset_id = a.asset_id
        JOIN Frequencies AS f ON p.service_frequency = f.service_frequency
    ) AS d
    WHERE d.next_due < ?
    GROUP BY d.location
    ORDER BY overdue_with_history + overdue_never_serviced DESC, d.location
    '''
proto_rows = cur.execute(sql, ('2026-11-02',)).fetchall()
print(len(proto_rows))
for location, with_history, never_serviced in proto_rows:
    print(f'{location}, {with_history}, {never_serviced}')

6
Pump Room 1, 7, 12
Substation, 5, 14
Boiler Room, 7, 9
Generator Room, 5, 9
Rooftop, 5, 6
Pump Room 2, 3, 7


In [32]:
rows = overdue_by_location(conn, '2026-11-02')
print(len(rows))
for location, with_history, never_serviced in rows:
    print(f'{location}, {with_history}, {never_serviced}')

6
Pump Room 1, 7, 12
Substation, 5, 14
Boiler Room, 7, 9
Generator Room, 5, 9
Rooftop, 5, 6
Pump Room 2, 3, 7


In [33]:
assert(proto_rows == overdue_by_location(conn, '2026-11-02'))

In [34]:
conn.close()